# Parsing agéntico de documentos

**Lección 5 · Clase 5.3** — la lección 4 extrajo un formulario de **una** página. El trabajo real son **PDFs largos, con tablas y gráficos**: informes, licitaciones, estados financieros, contratos.

Vamos a tomar **un mismo documento** y pasarlo por tres niveles, para ver con nuestros ojos qué compra cada peso extra:

| Nivel | Herramienta | Costo |
|---|---|---|
| **0** | `pypdf` — sacar el texto embebido | US$0 |
| **1** | DIY visual: rasterizar la página → `gpt-5-mini` la transcribe | centavos por página |
| **2** | **LlamaParse v2** — tier `fast` vs tier `agentic` | 1 vs 10 créditos por página |

La pregunta de negocio no es "cuál es mejor", sino **cuál necesito**: eso depende de volumen × complejidad × auditabilidad. Al final tendrás el criterio.

In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q pypdf==6.14.2 pypdfium2==5.12.1 pillow==12.3.0 langchain-openai==1.3.5 langchain-core==1.4.9 python-dotenv==1.2.2 llama-cloud==2.13.0
from dotenv import load_dotenv
import os

# Carga las llaves desde .env si existe (local); en Colab usa Secrets.
load_dotenv()

try:
    from google.colab import userdata  # type: ignore
    for llave in ("OPENAI_API_KEY", "LLAMA_CLOUD_API_KEY"):
        try:
            os.environ[llave] = userdata.get(llave) or os.environ.get(llave, "")
        except Exception:
            pass
except Exception:
    pass

HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
HAY_LLAMA = bool(os.environ.get("LLAMA_CLOUD_API_KEY"))
print("OPENAI_API_KEY presente:", HAY_OPENAI)
print("LLAMA_CLOUD_API_KEY presente:", HAY_LLAMA)
if not HAY_LLAMA:
    print("\n⚠️ El nivel 2 (LlamaParse) necesita una cuenta en https://cloud.llamaindex.ai")
    print("   Plan Free: 10.000 créditos/mes, sin tarjeta. Ponla en .env como LLAMA_CLOUD_API_KEY.")
    print("   Sin la llave el notebook corre igual: los niveles 0 y 1 funcionan y el 2 se salta.")

## El documento

`data/documento_ejemplo.pdf` — extracto del **capítulo II del IPoM de junio 2026** del Banco Central de Chile (*Evolución futura de la política monetaria*, págs. 35-43 del original). Son **9 páginas** con **4 tablas** y **7 gráficos**: exactamente la clase de documento que nadie quiere transcribir a mano.

> Fuente: Banco Central de Chile, Informe de Política Monetaria (IPoM), junio 2026 — https://www.bcentral.cl/publicaciones/politicas/informe-de-politica-monetaria-ipom (documento público; extracto preparado en julio 2026).

Trabajaremos sobre **la página 2**, que trae las tres cosas juntas: texto corrido, la **TABLA II.1** (supuestos del escenario internacional, con encabezados de dos niveles) y el **GRÁFICO II.1**.

In [ ]:
from pathlib import Path

pdf_path = Path("data/documento_ejemplo.pdf")
if not pdf_path.exists():
    import urllib.request
    # En Colab el archivo local no existe: se baja desde el repo (público)
    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(
        "https://raw.githubusercontent.com/josepenam/clases-diplomado-gen-ia/main/"
        "class_5_3_imagenes/leccion5_parsing_agentico/data/documento_ejemplo.pdf",
        headers={"User-Agent": "Mozilla/5.0 (clase-diplomado-gen-ia)"},
    )
    pdf_path.write_bytes(urllib.request.urlopen(req).read())

PAGINA = 2  # 1-indexada, como la ve un humano
print(f"Documento: {pdf_path} ({pdf_path.stat().st_size / 1e6:.1f} MB) — trabajaremos la página {PAGINA}")

## Nivel 0 — el texto embebido (`pypdf`)

Un PDF digital guarda los caracteres con sus coordenadas. `pypdf` los lee y los devuelve en orden aproximado de lectura. Gratis, instantáneo, sin API.

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
texto_plano = reader.pages[PAGINA - 1].extract_text()

print(texto_plano[:1400])

## ¿Qué pasó realmente?

**Sorpresa honesta: los números están ahí.** Este PDF es *nativo digital* (lo generó un programa, no un escáner), así que el texto se extrae bien. Primera lección: **no todo documento necesita IA** — si tus PDFs son así y solo quieres buscar palabras, `pypdf` basta y cuesta cero.

Pero mira de cerca y aparecen tres problemas que sí importan:

1. **El encabezado multinivel se aplastó.** La tabla tiene columnas `Prom. 10-19 · 2024 · 2025 · 2026 (f) · 2027 (f) · 2028 (f)`, pero salieron partidos en varias líneas. La fila dice `Términos de intercambio 1,0 3,3 7,6 4,2 0,2 0,9`: seis números sueltos. Tú puedes inferir qué año es cada uno; **un programa no**.
2. **Nada dice "esto es una tabla".** Sin estructura no hay `DataFrame`, no hay Excel, no hay carga a una base de datos. Solo texto.
3. **Los gráficos se vuelven sopa de números.** Las etiquetas de los ejes y las series salen como cifras sin contexto: información perdida, no degradada.

Y el caso que rompe todo: **si el PDF fuera escaneado, esta celda devolvería vacío** — no hay capa de texto que leer.

## Nivel 1 — DIY visual (la lección 4, recargada)

Mismo patrón de la lección 4, ahora sobre una página de informe: rasterizamos la página a imagen con `pypdfium2` (el motor de PDF de Chrome) y le pedimos a **gpt-5-mini** que la transcriba a **Markdown**. La diferencia clave con el nivel 0: le pedimos *estructura*, no caracteres.

In [ ]:
import base64
import io

import pypdfium2 as pdfium
from IPython.display import Markdown, display

pdf = pdfium.PdfDocument(pdf_path)
imagen = pdf[PAGINA - 1].render(scale=2.5).to_pil()  # scale=2.5 ≈ 180 dpi
pdf.close()

buf = io.BytesIO()
imagen.save(buf, format="PNG")
pagina_png = buf.getvalue()

print(f"Página {PAGINA} rasterizada: {imagen.size} px, {len(pagina_png) / 1e6:.1f} MB")
display(imagen.reduce(3))

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

INSTRUCCION = (
    "Transcribe esta página de un informe a Markdown, fiel al original y en español. "
    "Las tablas van como tablas Markdown, con los encabezados de año en la columna correcta. "
    "Para los gráficos, describe en una línea qué muestran y qué series contienen. "
    "No agregues comentarios ni interpretación: solo la transcripción."
)

if not HAY_OPENAI:
    markdown_diy = None
    print("⚠️ Falta OPENAI_API_KEY — se salta el nivel 1.")
else:
    llm = ChatOpenAI(model="gpt-5-mini")
    mensaje = HumanMessage(
        content=[
            {"type": "text", "text": INSTRUCCION},
            {
                "type": "image",
                "source_type": "base64",
                "data": base64.b64encode(pagina_png).decode("utf-8"),
                "mime_type": "image/png",
            },
        ]
    )
    markdown_diy = llm.invoke([mensaje]).content
    display(Markdown(markdown_diy))

## Nivel 1: lo que ganamos y por qué no basta

La tabla volvió **como tabla**: cada cifra bajo su año, lista para pegar en un Excel o parsear a un `DataFrame`. Y el gráfico ya no es sopa: el modelo lo *describe*. Esto es un salto real respecto del nivel 0.

Entonces, ¿por qué no dejarlo aquí y parsear los 500 PDFs de la empresa con este script? Porque en producción aparecen los problemas que no se ven con **una** página:

- **Orquestación:** una llamada por página, con reintentos, límites de tasa y control de costos. Lo escribes tú.
- **Orden de lectura entre páginas:** una tabla que cruza dos páginas, o notas al pie que continúan, se pierden si cada página va sola.
- **Alucinación silenciosa:** el modelo puede inventar una celda y devolverte un Markdown perfecto. No hay señal de error, y una tabla verosímil pero falsa es peor que un fallo evidente.
- **Sin auditoría:** ¿de qué parte de la página salió cada dato? Sin coordenadas ni confianza por campo, no hay revisión humana eficiente.

Resolver eso *es* el producto que llamamos **parsing agéntico**.

## Nivel 2 — parsing agéntico (LlamaParse v2)

Un servicio de parsing agéntico hace, **por página**, lo que nosotros hicimos a mano — y lo que no hicimos:

1. Extrae **las dos fuentes**: la capa de texto embebida del PDF *y* un screenshot de la página.
2. Las pasa a un **bucle agéntico** con un VLM que reconstruye layout y estructura, y **verifica su propia salida** (si algo no cuadra, reintenta esa página).
3. Devuelve Markdown de alta fidelidad — tablas, fórmulas, incluso diagramas.

Ese híbrido *texto + visión* es la razón por la que rinde mejor que "screenshot → LLM" a secas: cuando la capa de texto existe, la usa como verdad; cuando no (escaneos), cae en la visión.

**Los tiers de la API v2** (`fast` → `agentic_plus`) son la perilla precio/calidad. 1.000 créditos = US$1,25:

| Tier | Créditos por página | ≈ US$ por 1.000 páginas |
|---|---|---|
| `fast` | 1 | 1,25 |
| `cost_effective` | 3 | 3,75 |
| `agentic` | 10 | 12,50 |
| `agentic_plus` | 45 | 56 |

Documentación: https://developers.llamaindex.ai/llamaparse/ · El plan **Free** trae 10.000 créditos/mes sin tarjeta, y hay **caché de 48 h**: volver a parsear el mismo archivo no se cobra de nuevo.

In [ ]:
resultados = {}  # tier -> markdown de la página

if not HAY_LLAMA:
    print("⚠️ Falta LLAMA_CLOUD_API_KEY — se saltan los niveles 2a y 2b.")
    print("   Crea la cuenta free en https://cloud.llamaindex.ai (10.000 créditos/mes, sin tarjeta).")
else:
    from llama_cloud import LlamaCloud

    client = LlamaCloud()  # lee LLAMA_CLOUD_API_KEY del entorno

    # 1. Subir el archivo UNA vez y reutilizar su id en los dos tiers
    with open(pdf_path, "rb") as f:
        archivo = client.files.create(file=f, purpose="parse")
    print("file_id:", archivo.id)

    # 2. Tier fast — 1 crédito por página
    fast = client.parsing.parse(
        file_id=archivo.id,
        tier="fast",
        version="latest",
        expand=["markdown"],
    )
    resultados["fast"] = fast.markdown.pages[PAGINA - 1].markdown
    print(f"tier fast listo — {len(fast.markdown.pages)} páginas parseadas")

In [ ]:
if not HAY_LLAMA:
    print("⚠️ Falta LLAMA_CLOUD_API_KEY — se salta el nivel 2b.")
else:
    # Tier agentic — 10 créditos por página, sobre el MISMO file_id (no se re-sube nada)
    agentic = client.parsing.parse(
        file_id=archivo.id,
        tier="agentic",
        version="latest",
        expand=["markdown"],
    )
    resultados["agentic"] = agentic.markdown.pages[PAGINA - 1].markdown
    print(f"tier agentic listo — {len(agentic.markdown.pages)} páginas parseadas")
    display(Markdown(resultados["agentic"]))

## Los tres niveles, lado a lado

El momento de la verdad: la **misma página**, extraída de cuatro formas. Fíjate en la TABLA II.1 — ¿quedaron los años sobre la columna correcta? ¿sobrevivió el gráfico? ¿hay celdas inventadas?

In [ ]:
def encabezado(titulo: str) -> None:
    print("\n" + "=" * 70)
    print(titulo)
    print("=" * 70)


encabezado("NIVEL 0 — pypdf (texto embebido) · US$0")
print(texto_plano[:700])

if markdown_diy:
    encabezado("NIVEL 1 — DIY visual con gpt-5-mini · centavos por página")
    print(markdown_diy[:700])

for tier, creditos in (("fast", 1), ("agentic", 10)):
    if tier in resultados:
        encabezado(f"NIVEL 2 — LlamaParse tier {tier} · {creditos} crédito(s) por página")
        print(resultados[tier][:700])
    else:
        encabezado(f"NIVEL 2 — LlamaParse tier {tier}")
        print("(saltado: falta LLAMA_CLOUD_API_KEY)")

## Cierre: cómo se decide en una organización

**El mercado** (precios de lista, julio 2026 — cambian seguido, verifícalos):

| Servicio | Precio referencia | Para qué destaca |
|---|---|---|
| **LlamaParse** `fast` / `agentic` | US$1,25 / US$12,50 por 1.000 págs | Markdown de alta fidelidad; free tier generoso |
| **Mistral OCR 4** | ~US$4 por 1.000 págs (US$2 en batch) | Bounding boxes, confianza por palabra, y **self-hosted** (los datos no salen) |
| **Azure Document Intelligence** (Layout) | ~US$10 por 1.000 págs | Integración con el stack Microsoft; modelos prearmados (facturas, recibos) |
| **Google Document AI** | por procesador | Volumen alto en GCP |
| **DIY** (nivel 1) | centavos por página | Prototipos y formatos únicos donde nada calza |
| **LiteParse** (open source) | US$0 + tu infra | Parser local rápido, cuando nada puede salir de la empresa |

**El criterio: volumen × complejidad × auditabilidad.**

- **Volumen bajo, formato simple** → nivel 0 o 1. No pagues por un pipeline que no necesitas.
- **Volumen alto** → el precio por página manda, y el DIY deja de ser barato cuando sumas tu tiempo de orquestación.
- **Complejidad alta** (tablas cruzadas, escaneos, manuscrito) → tier agéntico; es donde el bucle *verificar y reintentar* se paga solo.
- **Auditabilidad** (regulado, financiero, salud) → exige citas y confianza por campo, para que un humano revise **solo** lo dudoso. LlamaExtract y Mistral OCR 4 devuelven coordenadas; el DIY no.
- **Datos que no pueden salir** → self-hosted (Mistral OCR) o local (LiteParse), aunque cueste más.

**Y la regla que no cambia:** un parser puede devolverte un JSON impecable con un dato equivocado. En montos, RUT, fechas y firmas, **validación determinista + revisión humana** — como en la lección 4. La siguiente pregunta natural es: *¿y si en vez de extraer el documento, lo busco por lo que se ve?* Eso es la **lección 6**.